In [ ]:
from scipy.optimize import minimize, Bounds, LinearConstraint
from scipy.stats import genpareto, kendalltau, rankdata
from scipy.cluster.hierarchy import linkage, cut_tree
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import adjusted_rand_score
from scipy.spatial.distance import squareform
from sklearn.metrics import silhouette_score
from dataclasses import dataclass, field
from sklearn.linear_model import Ridge
from joblib import Parallel, delayed
from datetime import datetime
from scipy.stats import ecdf
from arch import arch_model
from scipy.stats import qmc
import pyvinecopulib as pv
from fredapi import Fred
import yfinance as yf
import pandas as pd
import cvxpy as cp
import numpy as np
import time
import os

# Data Loader

In [ ]:
FACTOR_KINDS = {
  "Benchmark":    "compound",
  "Spread":       "compound",
  "USD":          "pct",
  "NFCI":         "diff",
  "BreakevenInf": "diff",
  "HighYieldOAS": "diff",
  "TIPS10Y":      "diff",
}
@dataclass
class FilterParams:
  min_length: tuple = (5, "y")
  debug: bool = True
  factor_kinds: dict = field(default_factory=lambda: dict(FACTOR_KINDS))
  week_anchor: str = "W-FRI"
  min_days_per_week: int = 3
  min_weeks: int = 26
  window_size: int = 104
  ridge_alpha: float = 1.0
  beta_change_window:int = 10
  tail_threshold:float=0.80
  tail_power:float=0.80
  min_effective_sample:int=30
  shrink_target_ess:int=100
  shrink:bool=True
  tau_shrinkage_params:float=0.20
  min_silhouette:float=0.10
  min_ari:float=0.65
  max_k:int=20

  def __post_init__(self):
    if not self.min_length[1] in ["d", "m", "y"]:
      raise ValueError(f"Invalid time period: {self.min_length[1]}")

    if self.min_length[0] <= 0:
      raise ValueError(f"Invalid min length: {self.min_length[0]}")

    if not 1 <= self.min_days_per_week <= 7:
      raise ValueError(f"Invalid min_days_per_week: {self.min_days_per_week}")

    if self.min_weeks <= 0:
      raise ValueError(f"Invalid min_weeks: {self.min_weeks}")

In [ ]:
class DataStore:
  def __init__(self, debug: bool = False):
    self.debug = debug

  def get_data(self, tickers: list, start: str, end: str, fred_api_key: str,
               interval: str = "1d", benchmark: str = "^GSPC",
               cyclical: str = "XLY", defensive: str = "XLP"):
    tmp = []
    for t in tickers:
      if isinstance(t, list):
        tmp.extend(t)
      elif isinstance(t, str):
        tmp.append(t)
      else:
        print(f"Warning: Skipping {t} | type: {type(t)}")
    tickers_clean = sorted(set(tmp))

    df_path = f"prices_{start}_{end}.parquet"
    if os.path.exists(df_path):
      df = pd.read_parquet(df_path)
    else:
      df = yf.download(tickers_clean, start=start, end=end, interval=interval)["Close"]
      df.to_parquet(df_path)

    returns = df.ffill().pct_change(fill_method=None)

    factors = self._get_factors(start, end, fred_api_key, benchmark, cyclical, defensive)
    return returns, factors

  def _fetch_equity_factors(self, start, end, benchmark="^GSPC",
                            cyclical="XLY", defensive="XLP"):
    r = yf.download([benchmark, "^IRX", cyclical, defensive],
                    start=start, end=end)["Close"]
    rets = r[[benchmark, cyclical, defensive]].pct_change(fill_method=None)
    rf_daily = (r["^IRX"].ffill() / 100) / 252

    factors = pd.DataFrame({
      "Benchmark": rets[benchmark] - rf_daily,
      "Spread": rets[cyclical] - rets[defensive],
    })
    return factors

  def _get_breakeven_inf(self, fred) -> pd.Series:
    return fred.get_series("DGS5") - fred.get_series("DFII5")

  def _fetch_fred(self, fred_api_key) -> pd.DataFrame:
    fred = Fred(api_key=fred_api_key)
    cols = {
      "NFCI":         fred.get_series("NFCI"),
      "USD":          fred.get_series("DTWEXBGS"),
      "TIPS10Y":      fred.get_series("DFII10"),
      "HighYieldOAS": fred.get_series("BAMLH0A0HYM2"),
      "BreakevenInf": self._get_breakeven_inf(fred),
    }
    out = pd.concat(cols, axis=1).sort_index()
    out.index = pd.to_datetime(out.index)
    return out

  def _get_factors(self, start, end, fred_api_key, benchmark="^GSPC",
                   cyclical="XLY", defensive="XLP") -> pd.DataFrame:
    path = f"factors_{start}_{end}.parquet"
    if os.path.exists(path):
      return pd.read_parquet(path)

    fred_df = self._fetch_fred(fred_api_key)
    eq_df = self._fetch_equity_factors(start, end, benchmark, cyclical, defensive)

    combined = pd.concat([fred_df, eq_df], axis=1, join="outer").sort_index()
    combined = combined.loc[start:end]
    combined.to_parquet(path)

    if self.debug:
      print("First valid date per factor:")
      print(combined.apply(pd.Series.first_valid_index))
    return combined

# Bootstrap

In [ ]:
@dataclass
class BootstrapParams:
    n_bootstraps: int = 200
    block_length: int = 10
    horizon: int | None = None
    circular: bool = False
    seed: int | None = None

In [ ]:
class BlockBootstrap:
    def __init__(self, bootstrap_params: BootstrapParams | None = None, **kwargs):
        self.p = bootstrap_params or BootstrapParams()
        self.rng = np.random.default_rng(self.p.seed)
        super().__init__(**kwargs)

    @staticmethod
    def _validate(returns: pd.DataFrame, params: BootstrapParams) -> int:
        if returns.isna().any().any():
            raise ValueError("returns contains NaNs; clean/align them first (e.g. dropna()).")

        T = len(returns)
        if params.block_length > T:
            raise ValueError(f"block_length ({params.block_length}) exceeds sample length ({T})")

        return params.horizon if params.horizon is not None else T

    @staticmethod
    def _sample_indices(
        T: int, horizon: int, params: BootstrapParams, rng: np.random.Generator,
    ) -> np.ndarray:
        L = params.block_length
        n_blocks = int(np.ceil(horizon / L))

        n_starts = T if params.circular else T - L + 1
        starts = rng.integers(0, n_starts, size=(params.n_bootstraps, n_blocks))

        idx = (starts[:, :, None] + np.arange(L)).reshape(params.n_bootstraps, -1)
        idx = idx[:, :horizon]

        return idx % T if params.circular else idx

    @staticmethod
    def resample(
        returns: pd.DataFrame,
        params: BootstrapParams,
        rng: np.random.Generator | None = None,
        as_dataframe: bool = False,
    ):
        rng = rng if rng is not None else np.random.default_rng(params.seed)
        horizon = BlockBootstrap._validate(returns=returns, params=params)
        values = returns.to_numpy()

        idx = BlockBootstrap._sample_indices(
            T=len(values), horizon=horizon, params=params, rng=rng,
        )
        samples = values[idx]

        if not as_dataframe:
            return samples

        index = pd.MultiIndex.from_product(
            [range(params.n_bootstraps), range(horizon)], names=["bootstrap", "step"]
        )
        return pd.DataFrame(
            samples.reshape(-1, samples.shape[2]), index=index, columns=returns.columns
        )

# FactorFilter

## FactorOptimizer

In [ ]:
class FactorOptimizer:
  def __init__(self, params, **kwargs):
    self.fp = params

  def _hedge_coverage_expr(z, hedge_score, p):
    N, S = hedge_score.shape
    H_s = []

    for s in range(S):
      h = hedge_score[:, s]

      levels = np.unique(
          np.r_[0.0, h]
      )

      coverage = 0
      for low, high in zip(levels[:-1], levels[1:]):
        eligible = np.flatnonzero(
            h >= high - 1e-12
        )

        coverage += (
            (high - low)
            * cp.minimum(
                1,
                cp.sum(
                    z[eligible]
                )
            )
        )

        H_s.append(coverage)

    return cp.sum(p * H_s)


  def optim_select_assets(self, unit, quality, r_hat, p_S, n_select, h_target=None):
    N_assets, K_factors = unit.shape
    I = np.identity(K_factors)
    z = cp.Variable(N_assets)
    h_scale = np.ones(N_assets) if h_target is None else h_target

    A = I + self.fp.factod_dir_importance * quality * cp.quad_form(unit, z)
    C_obj = cp.log_det(A)

    hedge_score = np.clip(
        r_hat / h_scale[None, :],
        0.0,
        1.0,
    )

    H_obj = self._hedge_coverage_expr(
        z,
        hedge_score,
        p_S
    )

    J_obj = C_obj + self.fp.hedge_theta * H_obj

    constraints = [
      z >= 0,
      z <= 1,
      cp.sum(z) == n_select
    ]

    obj = cp.Maximize()
    problem = cp.Problem(obj, constraints)

    problem.solve(solver=cp.CLARABEL)

    if problem.status not in (
      cp.OPTIMAL,
      cp.OPTIMAL_INACCURATE,
    ):
        raise RuntimeError(problem.status)

    return z.value, problem.value, A.value, H_obj.value




## FactorFilterHelpers

In [ ]:
class FactorFilterHelpers(FactorOptimizer, BlockBootstrap, Optimizer):
    def __init__(self, params: FilterParams, **kwargs):
        self.fp = params
        super().__init__(params=params, **kwargs)


    def _resample(self, returns: pd.DataFrame, as_dataframe: bool = False):
        return BlockBootstrap.resample(
            returns=returns, params=self.p, rng=self.rng, as_dataframe=as_dataframe,
        )

    def _fit_ridge(self, X: np.ndarray, y: np.ndarray):
        return Ridge(alpha=self.fp.ridge_alpha, fit_intercept=True).fit(X, y)

    def _bootstrap_coefs(self, Yb: np.ndarray, Xb: np.ndarray) -> np.ndarray:
        B, _, N = Yb.shape
        K = Xb.shape[2]
        coefs = np.empty((B, N, K), dtype=float)

        for b in range(B):
            scaler = StandardScaler()
            X_scaled = scaler.fit_transform(Xb[b])
            coefs[b] = (
                self._fit_ridge(X=X_scaled, y=Yb[b]).coef_
                / scaler.scale_[None, :]
            )

        return coefs

    def _select_assets_checked(
        self,
        unit: pd.DataFrame,
        quality: np.ndarray,
        r_hat: np.ndarray,
        n_select: int,
    ):
        N, K = unit.shape
        S = r_hat.shape[1]

        return self._validate_selection_result(
            result=self.optim_select_assets(
                unit=unit, quality=quality, r_hat=r_hat, n_select=n_select
            ),
            N=N, K=K, S=S, m=n_select,
        )

    @staticmethod
    def _naive(df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        df.index = pd.to_datetime(df.index).tz_localize(None)
        if not df.index.is_unique:
            raise ValueError("Time index must contain unique timestamps")
        return df.sort_index()

    def _compound_weekly(self, rets: pd.DataFrame) -> pd.DataFrame:
        if (rets.le(-1) & rets.notna()).any().any():
            raise ValueError("Compounded returns must be strictly greater than -1")

        g = np.log1p(rets).resample(self.fp.week_anchor)
        weekly = np.expm1(g.sum(min_count=1))
        counts = rets.resample(self.fp.week_anchor).count()

        return weekly.where(counts >= self.fp.min_days_per_week)

    def _to_weekly_returns(self, returns: pd.DataFrame) -> pd.DataFrame:
        return self._compound_weekly(rets=self._naive(df=returns))

    def _to_weekly_factors(self, factors: pd.DataFrame) -> pd.DataFrame:
        factors = self._naive(df=factors)
        kinds = self.fp.factor_kinds
        unknown = set(factors.columns) - set(kinds)

        if unknown:
            raise ValueError(f"No aggregation rule for factors: {sorted(unknown)}")

        supported = {"compound", "diff", "pct", "pct_change", "level"}
        invalid = {kinds[c] for c in factors if kinds[c] not in supported}

        if invalid:
            raise ValueError(f"Unsupported factor aggregation kind(s): {sorted(invalid)}")

        comp = [c for c in factors if kinds[c] == "compound"]
        levels = [c for c in factors if kinds[c] != "compound"]
        parts = []

        if comp:
            parts.append(self._compound_weekly(rets=factors[comp]))

        if levels:
            wk = factors[levels].resample(self.fp.week_anchor).last().ffill(limit=1)
            for c in levels:
                if kinds[c] == "diff":
                    wk[c] = wk[c].diff()

                elif kinds[c] in {"pct", "pct_change"}:
                    wk[c] = wk[c].pct_change(fill_method=None)

            parts.append(wk)

        if not parts:
            raise ValueError("At least one factor is required")
        return pd.concat(parts, axis=1).loc[:, factors.columns]

    def _time_horizon_filter(self, returns: pd.DataFrame) -> pd.DataFrame:
        if returns.empty:
            raise ValueError("No return observations")

        end = pd.to_datetime(returns.index.max())
        min_l, tp = self.fp.min_length

        if tp not in {"y", "m", "d"}:
            raise ValueError("min_length time unit must be y, m, or d")

        min_start = end - pd.DateOffset(
            years=min_l if tp == "y" else 0,
            months=min_l if tp == "m" else 0,
            days=min_l if tp == "d" else 0,
        )

        if self.fp.debug:
            print(f"Min start: {min_start}")

        keep = [
            asset for asset in returns
            if not returns[asset].dropna().empty
            and pd.to_datetime(returns[asset].dropna().index.min()) <= min_start
        ]

        return returns.loc[:, keep].dropna()

    def _rolling_factor_params(self, returns: pd.DataFrame, factors: pd.DataFrame):
        idx = returns.index.intersection(factors.index).sort_values()
        returns, factors = returns.loc[idx], factors.loc[idx]

        if returns.isna().any().any() or factors.isna().any().any():
            raise ValueError("Missing data in regression inputs")

        X, Y = factors.to_numpy(dtype=float), returns.to_numpy(dtype=float)
        T, N = Y.shape
        K = X.shape[1]
        w = self.fp.window_size

        if T <= w:
            raise ValueError(f"Need more than {w} weekly observations; got {T}")

        betas_raw = np.full((T, N, K), np.nan)
        betas_std = np.full((T, N, K), np.nan)
        delta_r2 = np.full((T, N, K), np.nan)
        r2_in = np.full((T, N), np.nan)
        r2_oos = np.full((T, N), np.nan)
        rmse_oos = np.full((T, N), np.nan)
        resid_vol_in = np.full((T, N), np.nan)
        oos_resid = np.full((T, N), np.nan)

        cumulative_sse = np.zeros(N)
        cumulative_sst = np.zeros(N)
        cumulative_sse_wo = np.zeros((K, N))

        def safe_r2(sse, sst):
            ratio = np.full_like(sse, np.nan, dtype=float)
            np.divide(sse, sst, out=ratio, where=sst > 0)
            return 1.0 - ratio

        def fit_window(start, stop):
            x_train, y_train = X[start:stop], Y[start:stop]
            scaler = StandardScaler()
            x_scaled = scaler.fit_transform(x_train)
            model = self._fit_ridge(X=x_scaled, y=y_train)
            train_resid = y_train - model.predict(x_scaled)
            y_mean = y_train.mean(axis=0)
            train_sse = np.sum(train_resid**2, axis=0)
            train_sst = np.sum((y_train - y_mean)**2, axis=0)
            return (
                model, scaler, safe_r2(sse=train_sse, sst=train_sst),
                train_resid.std(axis=0, ddof=0), y_mean,
            )

        for t in range(w, T):
            model, scaler, train_r2, train_vol, y_mean = fit_window(start=t - w, stop=t)
            betas_std[t - 1] = model.coef_
            betas_raw[t - 1] = model.coef_ / scaler.scale_[None, :]
            r2_in[t - 1], resid_vol_in[t - 1] = train_r2, train_vol

            x_test = scaler.transform(X[t:t + 1])
            prediction = model.predict(x_test)[0]
            error = Y[t] - prediction
            oos_resid[t] = error
            cumulative_sse += error**2
            cumulative_sst += (Y[t] - y_mean)**2

            x_win, y_win = scaler.transform(X[t - w:t]), Y[t - w:t]

            for k in range(K):
                if K > 1:
                    x_wo = np.delete(x_win, k, axis=1)
                    model_wo = self._fit_ridge(X=x_wo, y=y_win)
                    pred_wo = model_wo.predict(np.delete(x_test, k, axis=1))[0]

                else:
                    pred_wo = y_mean
                cumulative_sse_wo[k] += (Y[t] - pred_wo)**2

            n_oos = t - w + 1
            r2_oos[t] = safe_r2(sse=cumulative_sse, sst=cumulative_sst)
            rmse_oos[t] = np.sqrt(cumulative_sse / n_oos)
            delta = np.full((K, N), np.nan)
            np.divide(
                cumulative_sse_wo - cumulative_sse,
                cumulative_sst,
                out=delta,
                where=cumulative_sst > 0,
            )
            delta_r2[t] = delta.T

        model, scaler, train_r2, train_vol, _ = fit_window(start=T - w, stop=T)
        betas_std[-1] = model.coef_
        betas_raw[-1] = model.coef_ / scaler.scale_[None, :]
        r2_in[-1], resid_vol_in[-1] = train_r2, train_vol

        residuals_df = pd.DataFrame(oos_resid, index=idx, columns=returns.columns)
        resid_vol_oos = (
            residuals_df.expanding(min_periods=2).std(ddof=0).to_numpy()
        )

        beta_columns = pd.MultiIndex.from_product(
            [returns.columns, factors.columns], names=["Asset", "Factor"]
        )

        def make_beta_df(values):
            return pd.DataFrame(
                values.reshape(T, N * K), index=idx, columns=beta_columns
            )

        metrics = ["R2_in", "R2_oos", "RMSE_oos", "ResidVol_in", "ResidVol_oos"]
        stats_values = np.stack(
            [r2_in, r2_oos, rmse_oos, resid_vol_in, resid_vol_oos], axis=2
        )

        stats_df = pd.DataFrame(
            stats_values.reshape(T, N * len(metrics)),
            index=idx,
            columns=pd.MultiIndex.from_product(
                [returns.columns, metrics], names=["Asset", "Metric"]
            ),
        )

        return (
            make_beta_df(values=betas_raw),
            make_beta_df(values=betas_std),
            make_beta_df(values=delta_r2),
            stats_df,
            residuals_df,
        )

    @staticmethod
    def _shrink_vasicek(
      betas_raw: pd.DataFrame,
      est_var: pd.DataFrame,
    ) -> tuple[pd.DataFrame, pd.DataFrame]:
        est_var = est_var.reindex(index=betas_raw.index, columns=betas_raw.columns)

        if est_var.isna().any().any() or (est_var < 0).any().any():
            raise ValueError("est_var must contain nonnegative (N,K) estimates")

        prior_mean = betas_raw.mean(axis=0)

        prior_var = (
            betas_raw.var(axis=0, ddof=1).fillna(0.0)
            - est_var.mean(axis=0)
        ).clip(lower=0.0)

        numerator = np.broadcast_to(prior_var.to_numpy()[None, :], betas_raw.shape)
        denominator = numerator + est_var.to_numpy()

        weights = np.divide(
            numerator, denominator,
            out=np.zeros_like(denominator), where=denominator > 0,
        )

        w_raw = pd.DataFrame(weights, index=betas_raw.index, columns=betas_raw.columns)
        adjusted = w_raw * betas_raw + (1.0 - w_raw) * prior_mean

        return adjusted, w_raw

    @staticmethod
    def _sqrt_psd(cov: np.ndarray) -> np.ndarray:
        cov = np.asarray(cov, dtype=float)

        if cov.ndim not in (2, 3) or cov.shape[-1] != cov.shape[-2]:
            raise ValueError("cov must be (K,K) or (B,K,K)")

        cov = (cov + np.swapaxes(cov, -1, -2)) / 2.0
        eigval, eigvec = np.linalg.eigh(cov)
        root = np.sqrt(np.clip(eigval, 0.0, None))

        return (eigvec * root[..., None, :]) @ np.swapaxes(eigvec, -1, -2)

    def _risk_exposure_similarity(
      self, betas: pd.DataFrame, factors: pd.DataFrame,
    ):
        if isinstance(betas, pd.Series):
            betas = betas.unstack("Factor")

        if not isinstance(betas, pd.DataFrame) or isinstance(betas.columns, pd.MultiIndex):
            raise ValueError("betas must be (N,K), indexed by Asset and columns by Factor")

        missing = [c for c in factors.columns if c not in betas.columns]

        if missing:
            raise ValueError(f"No betas for factors: {missing}")

        betas = betas.loc[:, factors.columns]
        sigma = factors.dropna().cov().loc[factors.columns, factors.columns].to_numpy()
        sqrt_sigma = self._sqrt_psd(cov=sigma)
        exposure = betas.to_numpy(dtype=float) @ sqrt_sigma
        norms = np.linalg.norm(exposure, axis=1, keepdims=True)

        if np.any(norms <= 1e-12):
            raise ValueError("Cannot normalize an asset with zero factor exposure")

        U = exposure / norms
        cos = np.clip(U @ U.T, -1.0, 1.0)
        exposures = pd.DataFrame(exposure, index=betas.index, columns=factors.columns)

        unit = pd.DataFrame(U, index=betas.index, columns=factors.columns)
        cos_sim = pd.DataFrame(cos, index=betas.index, columns=betas.index)
        cos_sim.index.name = cos_sim.columns.name = "Asset"

        return exposures, unit, cos_sim

    def _get_h_ref(
      self,
      betas: np.ndarray,
      factor_cov: np.ndarray,
      factor_scenarios: np.ndarray,
      a: np.ndarray,
      scenario_idx=None,
      i_idx=None,
      j_idx=None,
    ) -> np.ndarray:
        beta = np.asarray(betas, dtype=float)

        if beta.ndim != 2:
            raise ValueError("betas must have shape (N,K)")

        N, K = beta.shape
        delta = np.asarray(factor_scenarios, dtype=float)
        scale = np.asarray(a, dtype=float)
        sigma = np.asarray(factor_cov, dtype=float)

        if delta.ndim != 2 or delta.shape[1] != K or sigma.shape != (K, K):
            raise ValueError("factor_scenarios (S,K), betas (N,K), covariance (K,K) required")

        if scale.shape != (len(delta),):
            raise ValueError("a must have shape (S,)")

        if scenario_idx is not None:
            s = np.asarray(scenario_idx, dtype=int)
            delta, scale = delta[s], scale[s]

        if not np.isfinite(scale).all() or np.any(scale <= 0):
            raise ValueError("Scenario scales must be positive and finite")

        i_idx = np.arange(N) if i_idx is None else np.atleast_1d(i_idx).astype(int)
        j_idx = np.arange(N) if j_idx is None else np.atleast_1d(j_idx).astype(int)

        mag = np.linalg.norm(beta @ self._sqrt_psd(cov=sigma), axis=1)

        ratio = np.divide(
            mag[i_idx, None], mag[None, j_idx],
            out=np.zeros((len(i_idx), len(j_idx))),
            where=mag[None, j_idx] > 1e-12,
        )

        proj = beta[j_idx] @ delta.T
        h = ratio[:, :, None] * proj[None, :, :] / scale[None, None, :]

        return np.clip(h, 0.0, 1.0)

    def _get_ref_redundancy(
        self,
        unit: pd.DataFrame,
        A_prev: np.ndarray,
        H: np.ndarray,
        quality: np.ndarray,
        prev_mask: np.ndarray,
        new_idx: int,
        h_ref: np.ndarray,
        p_s: np.ndarray,
        cost: np.ndarray | None = None,
    ) -> float:
        old = np.flatnonzero(prev_mask)
        if old.size == 0:
            return 0.0

        U = unit.to_numpy(dtype=float)
        N, K = U.shape
        q = np.asarray(quality, dtype=float)

        if q.shape != (N,) or A_prev.shape != (K, K):
            raise ValueError("quality (N,), A_prev (K,K) required")

        H = np.asarray(H, dtype=float)
        p_s = np.asarray(p_s, dtype=float)
        h = np.asarray(h_ref, dtype=float)

        if h.shape != (len(old), len(H)) or p_s.shape != H.shape:
            raise ValueError("h_ref must be (n_previous,S), and H/p_s must be (S,)")

        kappa = self.fp.factor_dir_importance
        theta = self.fp.theta
        lam_c = self.fp.lambda_cost

        U_j = U[old]
        quad = np.einsum("jk,kj->j", U_j, np.linalg.solve(A_prev, U_j.T))

        info = np.log1p(kappa * q[new_idx] * quad)
        scen = theta * (np.maximum(0.0, h - H[None, :]) @ p_s)
        c_i = 0.0 if cost is None else float(np.asarray(cost)[new_idx])

        return float((info + scen - lam_c * c_i).max())

    def _get_excess_coverage(
        self,
        unit: pd.DataFrame,
        betas: np.ndarray,
        factor_cov: np.ndarray,
        quality: np.ndarray,
        factor_scenarios: np.ndarray,
        scenario_scale: np.ndarray,
        p_s: np.ndarray,
        prev_result,
        current_result,
        cost: np.ndarray | None = None,
    ) -> tuple[float, float, float]:
        _, J_prev, prev_mask, A_prev, H_prev = prev_result
        _, J_curr, selected_mask, _, _ = current_result
        new = np.flatnonzero(selected_mask & ~prev_mask)
        removed = np.flatnonzero(prev_mask & ~selected_mask)

        if len(new) != 1 or len(removed) != 0:
            raise ValueError(
                "_select_assets must return nested selected sets; "
                "one new asset with no removals is required"
            )

        new_idx = int(new[0])
        old = np.flatnonzero(prev_mask)

        if len(old):
            h_ref = self._get_h_ref(
                betas=betas,
                factor_cov=factor_cov,
                factor_scenarios=factor_scenarios,
                a=scenario_scale,
                i_idx=[new_idx],
                j_idx=old,
            )[0]

        else:
            h_ref = np.empty((0, len(p_s)), dtype=float)

        redundancy = self._get_ref_redundancy(
            unit=unit,
            A_prev=A_prev,
            H=H_prev,
            quality=quality,
            prev_mask=prev_mask,
            new_idx=new_idx,
            h_ref=h_ref,
            p_s=p_s,
            cost=cost,
        )

        marginal_gain = J_curr - J_prev

        return float(marginal_gain - redundancy), float(redundancy), float(marginal_gain)

    @staticmethod
    def _validate_selection_result(result, N: int, K: int, S: int, m: int):
        if len(result) != 5:
            raise ValueError("_select_assets must return (weights, J, mask, A, H)")

        weights, J, mask, A, H = result
        weights = np.asarray(weights, dtype=float)
        mask = np.asarray(mask, dtype=bool)
        A = np.asarray(A, dtype=float)
        H = np.asarray(H, dtype=float)

        if weights.shape != (N,) or mask.shape != (N,):
            raise ValueError("Optimizer weights/mask must both be (N,)")

        if A.shape != (K, K) or H.shape != (S,):
            raise ValueError("Optimizer A must be (K,K) and H must be (S,)")

        if np.count_nonzero(mask) != m:
            raise ValueError(f"Optimizer returned {mask.sum()} assets; requested {m}")

        if not np.isfinite(J) or not (np.isfinite(A).all() and np.isfinite(H).all()):
            raise ValueError("Optimizer returned non-finite results")

        return weights, float(J), mask, A, H

## FactorFilter

In [ ]:
class FactorFilter(FactorFilterHelpers):
    def __init__(
        self,
        filter_params: FilterParams,
        bootstrap_params: BootstrapParams,
    ):
        super().__init__(params=filter_params, bootstrap_params=bootstrap_params)

    def _get_bootstraps(self, returns: pd.DataFrame, factors: pd.DataFrame):
        common = returns.dropna().index.intersection(factors.dropna().index)
        Y_df, X_df = returns.loc[common], factors.loc[common]
        if len(common) < 2:
            raise ValueError("Need at least two aligned weekly observations")

        assets, fnames = Y_df.columns, X_df.columns
        N, K = len(assets), len(fnames)

        draws = np.asarray(
            self._resample(returns=pd.concat([Y_df, X_df], axis=1)), dtype=float
        )

        if draws.ndim != 3 or draws.shape[2] != N + K:
            raise ValueError("resample must return (B, W, N+K) in input column order")
        B, W, _ = draws.shape

        if B < 2 or W < 2:
            raise ValueError("Need B>=2 bootstrap draws with W>=2 observations each")

        if not np.isfinite(draws).all():
            raise ValueError("Bootstrap draws contain non-finite values")

        Yb = draws[:, :, :N]
        Xb = draws[:, :, N:].copy()

        coefs = self._bootstrap_coefs(Yb=Yb, Xb=Xb)

        lower, upper = np.percentile(coefs, [2.5, 97.5], axis=0)
        p_pos = (coefs > 0).mean(axis=0)

        stability = np.maximum(p_pos, 1 - p_pos)
        factor_index = pd.Index(fnames, name="Factor")

        percentile_intervals = pd.DataFrame(
            np.stack([lower, upper], axis=2).transpose(1, 0, 2).reshape(K, N * 2),
            index=factor_index,
            columns=pd.MultiIndex.from_product(
                [assets, ["Lower CI (2.5%)", "Upper CI (97.5%)"]],
                names=["Asset", "Bound"],
            ),
        )

        sign_stability = pd.DataFrame(
            stability.T, index=factor_index,
            columns=pd.Index(assets, name="Asset"),
        )

        beta_estimation_var = coefs.var(axis=0, ddof=1)

        return (
          percentile_intervals,
          sign_stability,
          p_pos,
          beta_estimation_var,
          coefs,
          Xb,
        )

    def _get_factor_exposure_features(
      self,
      returns: pd.DataFrame,
      factors: pd.DataFrame,
    ):
        weekly_returns = self._to_weekly_returns(returns=returns).dropna()
        weekly_factors = self._to_weekly_factors(factors=factors).dropna()
        common = weekly_returns.index.intersection(weekly_factors.index).sort_values()
        X, Y = weekly_factors.loc[common], weekly_returns.loc[common]

        if len(common) <= self.fp.window_size:
            raise ValueError("Insufficient aligned weekly data for rolling estimation")

        w = self.fp.window_size
        (
            percentile_intervals, sign_stability, p_pos,
            beta_estimation_var, coefs, factor_bootstrap,
        ) = self._get_bootstraps(returns=Y.iloc[-w:], factors=X.iloc[-w:])

        betas_raw, betas_std, delta_r2, stats_df, residuals = (
            self._rolling_factor_params(returns=Y, factors=X)
        )
        assets, fnames = Y.columns, X.columns
        est_var = pd.DataFrame(beta_estimation_var, index=assets, columns=fnames)

        latest_raw = betas_raw.iloc[-1].unstack("Factor").reindex(
            index=assets, columns=fnames
        )

        beta_adjusted, _ = self._shrink_vasicek(betas_raw=latest_raw, est_var=est_var)

        lag = int(self.fp.beta_change_window)

        if lag < 1:
            raise ValueError("beta_change_window must be >= 1")

        if len(betas_raw) > lag and betas_raw.iloc[-1 - lag].notna().all():
            old_raw = betas_raw.iloc[-1 - lag].unstack("Factor").reindex(
                index=assets, columns=fnames
            )
            old_adjusted, _ = self._shrink_vasicek(
                betas_raw=old_raw, est_var=est_var
            )
            beta_change = beta_adjusted - old_adjusted

        else:
            beta_change = pd.DataFrame(np.nan, index=assets, columns=fnames)

        last_delta = delta_r2.iloc[-1].unstack("Factor").reindex(
            index=assets, columns=fnames
        )

        factor_part = pd.concat(
          {
            f: pd.DataFrame(
              {
                "Lower CI (2.5%)": percentile_intervals.loc[f].xs(
                  "Lower CI (2.5%)", level="Bound"
                ).reindex(assets),
                "Upper CI (97.5%)": percentile_intervals.loc[f].xs(
                  "Upper CI (97.5%)", level="Bound"
                ).reindex(assets),
                "SignStability": sign_stability.loc[f].reindex(assets),
                "DeltaR2_oos": last_delta[f],
              }, index=assets,
            )
            for f in fnames
          }, axis=1,
        )
        model_cols = ["R2_in", "R2_oos", "RMSE_oos", "ResidVol_in", "ResidVol_oos"]
        latest_stats = stats_df.iloc[-1].unstack("Metric").reindex(index=assets)
        model_part = pd.concat({"Model": latest_stats[model_cols]}, axis=1)
        summary = pd.concat([factor_part, model_part], axis=1)
        summary.columns.names = ["Factor", "Field"]
        summary.index.name = "Asset"

        return (
            beta_adjusted, beta_change, summary,
            coefs, factor_bootstrap, X.iloc[-w:], est_var,
        )

    def _get_uncertainty(
      self,
      coefs: np.ndarray,
      factors_boot: np.ndarray,
      unit: pd.DataFrame,
      quality: np.ndarray,
      m: int,
      factor_scenarios: np.ndarray,
      scenario_scale: np.ndarray,
      p_s: np.ndarray,
      beta_estimation_var: pd.DataFrame | None = None,
    ) -> float:
        coefs = np.asarray(coefs, dtype=float)
        factors_boot = np.asarray(factors_boot, dtype=float)

        if coefs.ndim != 3 or factors_boot.ndim != 3:
            raise ValueError("coefs (B,N,K) and factors_boot (B,W,K) required")

        B, N, K = coefs.shape
        if B < 2 or factors_boot.shape[0] != B or factors_boot.shape[2] != K:
            raise ValueError("Incompatible bootstrap draws (B,N,K) and (B,W,K)")

        if factors_boot.shape[1] < 2 or not 2 <= m <= N:
            raise ValueError("Need W>=2 and m in [2,N]")

        if unit.shape != (N, K) or quality.shape != (N,):
            raise ValueError("unit (N,K) and quality (N,) must match coefs")

        Xc = factors_boot - factors_boot.mean(axis=1, keepdims=True)
        cov_b = np.einsum("bti,btj->bij", Xc, Xc) / (factors_boot.shape[1] - 1)
        sqrt_b = self._sqrt_psd(cov=cov_b)
        excess = np.empty(B, dtype=float)
        for b in range(B):
            beta_b = pd.DataFrame(coefs[b], index=unit.index, columns=unit.columns)

            if beta_estimation_var is not None:
                beta_b, _ = self._shrink_vasicek(
                    betas_raw=beta_b, est_var=beta_estimation_var
                )

            beta_array = beta_b.to_numpy(dtype=float)
            exposure_b = beta_array @ sqrt_b[b]
            norms = np.linalg.norm(exposure_b, axis=1, keepdims=True)

            if np.any(norms <= 1e-12):
                raise ValueError(f"Bootstrap {b} contains zero factor exposure")

            unit_b = pd.DataFrame(
                exposure_b / norms, index=unit.index, columns=unit.columns
            )

            r_hat_b = beta_array @ factor_scenarios.T
            prev = self._select_assets_checked(
                unit=unit_b, quality=quality, r_hat=r_hat_b, n_select=m - 1,
            )

            current = self._select_assets_checked(
                unit=unit_b, quality=quality, r_hat=r_hat_b, n_select=m,
            )

            excess[b], _, _ = self._get_excess_coverage(
                unit=unit_b,
                betas=beta_array,
                factor_cov=cov_b[b],
                quality=quality,
                factor_scenarios=factor_scenarios,
                scenario_scale=scenario_scale,
                p_s=p_s,
                prev_result=prev,
                current_result=current,
            )

        return float(excess.std(ddof=1))

    def _select_opt_n_assets(
      self,
      beta_adjusted: pd.DataFrame,
      unit: pd.DataFrame,
      quality: np.ndarray,
      coefs: np.ndarray,
      factor_scenarios: np.ndarray,
      scenario_scale: np.ndarray,
      factor_bootstrap: np.ndarray,
      factor_cov: pd.DataFrame,
      beta_estimation_var: pd.DataFrame,
      min_assets: int = 2,
      scenario_idx=None,
    ) -> np.ndarray:
        N, K = unit.shape
        if not 1 <= min_assets <= N:
            raise ValueError("min_assets must be in [1,N]")

        if not beta_adjusted.index.equals(unit.index) or not beta_adjusted.columns.equals(unit.columns):
            raise ValueError("beta_adjusted and unit must have identical (Asset,Factor) axes")

        if coefs.shape[1:] != (N, K):
            raise ValueError("coefs must be (B,N,K) with unit asset/factor ordering")

        if quality.shape != (N,) or not np.isfinite(quality).all() or np.any(quality < 0):
            raise ValueError("quality must be nonnegative finite (N,)")

        if not isinstance(factor_cov, pd.DataFrame):
            raise ValueError("factor_cov must be a DataFrame indexed by Factor")

        sigma = factor_cov.loc[unit.columns, unit.columns].to_numpy(dtype=float)

        delta = np.asarray(factor_scenarios, dtype=float)
        a = np.asarray(scenario_scale, dtype=float)

        if delta.ndim != 2 or delta.shape[1] != K or a.shape != (len(delta),):
            raise ValueError("factor_scenarios (S,K) and scenario_scale (S,) required")

        if not np.isfinite(delta).all() or not np.isfinite(a).all() or np.any(a <= 0):
            raise ValueError("Scenario factor shocks must be finite; scales must be >0")

        S_all = len(delta)
        if S_all == 0:
            raise ValueError("At least one scenario is required")

        if scenario_idx is not None:
            selected = np.asarray(scenario_idx)
            if selected.dtype == bool:
                if selected.shape != (S_all,):
                    raise ValueError("Boolean scenario_idx must be (S,)")
            else:
                selected = np.atleast_1d(selected).astype(int)
                if selected.ndim != 1 or np.any((selected < 0) | (selected >= S_all)):
                    raise ValueError("scenario_idx contains invalid positions")
            delta, a = delta[selected], a[selected]

        if len(delta) == 0:
            raise ValueError("At least one selected scenario is required")

        S = len(delta)

        p_s = np.full(S, 1.0 / S)

        beta_array = beta_adjusted.to_numpy(dtype=float)
        r_hat = beta_array @ delta.T

        baseline = self._select_assets_checked(
            unit=unit, quality=quality, r_hat=r_hat, n_select=min_assets,
        )

        opt_selection = baseline[0].copy()
        prev = baseline

        for m in range(min_assets + 1, N + 1):
            current = self._select_assets_checked(
                unit=unit, quality=quality, r_hat=r_hat, n_select=m,
            )

            excess, redundancy, marginal_gain = self._get_excess_coverage(
                unit=unit,
                betas=beta_array,
                factor_cov=sigma,
                quality=quality,
                factor_scenarios=delta,
                scenario_scale=a,
                p_s=p_s,
                prev_result=prev,
                current_result=current,
            )

            uncertainty = self._get_uncertainty(
                coefs=coefs,
                factors_boot=factor_bootstrap,
                unit=unit,
                quality=quality,
                m=m,
                factor_scenarios=delta,
                scenario_scale=a,
                p_s=p_s,
                beta_estimation_var=beta_estimation_var,
            )

            if self.fp.debug:
                print(
                  f"m={m}: marginal_gain={marginal_gain:.6g}, "
                  f"redundancy={redundancy:.6g}, excess={excess:.6g}, "
                  f"bootstrap_sd={uncertainty:.6g}"
                )

            if excess <= uncertainty:
                break

            opt_selection = current[0].copy()
            prev = current

        return opt_selection

    def select_assets(
      self,
      returns: pd.DataFrame,
      factors: pd.DataFrame,
      quality: pd.Series | np.ndarray,
      factor_scenarios: pd.DataFrame | np.ndarray,
      scenario_scale: np.ndarray,
      min_assets: int = 2,
      scenario_idx=None,
    ) -> np.ndarray:
        (
            beta_adjusted, beta_change, summary, coefs,
            factor_bootstrap, aligned_factor_window, est_var,
        ) = self._get_factor_exposure_features(returns=returns, factors=factors)

        assets, fnames = beta_adjusted.index, beta_adjusted.columns

        if isinstance(quality, pd.Series):
            quality_array = quality.reindex(assets).to_numpy(dtype=float)

        else:
            quality_array = np.asarray(quality, dtype=float)

        if quality_array.shape != (len(assets),) or not np.isfinite(quality_array).all():
            raise ValueError("quality must align with surviving asset columns")

        if isinstance(factor_scenarios, pd.DataFrame):
            missing = fnames.difference(factor_scenarios.columns)
            if len(missing):
                raise ValueError(f"Missing scenario factors: {missing.tolist()}")
            factor_scenarios = factor_scenarios.loc[:, fnames].to_numpy(dtype=float)

        _, unit, _ = self._risk_exposure_similarity(
            betas=beta_adjusted, factors=aligned_factor_window
        )
        factor_cov = aligned_factor_window.cov()

        return self._select_opt_n_assets(
            beta_adjusted=beta_adjusted,
            unit=unit,
            quality=quality_array,
            coefs=coefs,
            factor_scenarios=factor_scenarios,
            scenario_scale=scenario_scale,
            factor_bootstrap=factor_bootstrap,
            factor_cov=factor_cov,
            beta_estimation_var=est_var,
            min_assets=min_assets,
            scenario_idx=scenario_idx,
        )

    def factor_filter(
      self,
      returns: pd.DataFrame,
      factors: pd.DataFrame,
      quality: pd.Series | np.ndarray,
      factor_scenarios: pd.DataFrame | np.ndarray,
      scenario_scale: np.ndarray,
      min_assets: int = 2,
      scenario_idx=None,
    ) -> pd.Series:
        if not isinstance(quality, pd.Series):
            quality = pd.Series(np.asarray(quality, dtype=float), index=returns.columns)

        screened = self._time_horizon_filter(returns=returns)

        selection = self.select_assets(
            returns=screened,
            factors=factors,
            quality=quality,
            factor_scenarios=factor_scenarios,
            scenario_scale=scenario_scale,
            min_assets=min_assets,
            scenario_idx=scenario_idx,
        )

        return pd.Series(selection, index=screened.columns, name="Weight")

# HierarchicalFilter

## TailWeightedKendall

In [ ]:
class TailWeightedKendall:
  _BLOCK_ELEMS = 2e7

  def __init__(self, params, **kwargs):
    self.p = params

  def empirical_uniforms(self, losses, T):
    losses = np.asarray(losses, dtype=float)
    if losses.ndim not in (2, 3):
      raise ValueError("losses must be (T, N) or (B, T, N).")

    k_ranks = rankdata(
        losses, method="average", axis=-2
    )
    return k_ranks / (T + 1.0)

  def tail_weight(self, u):
    u = np.asarray(u, dtype=float)
    tail_depth = np.maximum(
        (u - self.p.tail_threshold) / (1 - self.p.tail_threshold),
        0.0
    ) ** self.p.tail_power

    return tail_depth

  def _get_tau(self, U, w):
    U = np.asarray(U, dtype=float)
    w = np.asarray(w, dtype=float)
    B, T_dim, N = U.shape
    g = 1.0 - w

    G = np.zeros((B, N, N))
    D_and = np.zeros((B, N, N))
    D_or = np.zeros((B, N, N))
    Gw_and = np.zeros((B, N, N))
    Gw_or = np.zeros((B, N, N))
    zeros = np.zeros((B, N))

    block_size = max(1, int(self._BLOCK_ELEMS // (B * T_dim * N)))
    for a in range(0, T_dim, block_size):
      b = min(a + block_size, T_dim)

      S = np.sign(U[:, a:b, None, :] - U[:, None, :, :])
      S2 = S * S
      S_flat = S.reshape(B, -1, N)

      G += S_flat.swapaxes(1, 2) @ S_flat
      zeros += (S_flat == 0).sum(axis=1)

      h_t = w[:, a:b, None, :]
      h_s = w[:, None, :, :]
      W_pairwise = h_t * h_s

      A_flat = (S * W_pairwise).reshape(B, -1, N)
      Gw_and += A_flat.swapaxes(1, 2) @ A_flat
      S2W_flat = (S2 * W_pairwise).reshape(B, -1, N)
      D_and += 0.5 * (S2W_flat.swapaxes(1, 2) @ W_pairwise.reshape(B, -1, N))

      mask = (np.arange(T_dim)[None, :] < np.arange(a, b)[:, None])[None, :, :, None]
      Sm = S * mask
      Sm2 = S2 * mask
      g_t = g[:, a:b, None, :]
      g_s = g[:, None, :, :]

      terms = (
          (h_s * h_t, g_s * g_t),
          (h_s,       g_s * h_t),
          (h_t,       h_s * g_t),
          (None,      h_s * h_t),
      )
      for x, y in terms:
        Sx = Sm if x is None else Sm * x
        Sx2 = Sm2 if x is None else Sm2 * x
        Sy = (Sm * y).reshape(B, -1, N)
        y_flat = np.broadcast_to(y, S.shape).reshape(B, -1, N)

        Gw_or += Sx.reshape(B, -1, N).swapaxes(1, 2) @ Sy
        D_or += Sx2.reshape(B, -1, N).swapaxes(1, 2) @ y_flat

    return G, Gw_and, Gw_or, zeros, D_and, D_or

  def _adjust_tau_or(self, tau_or_raw):
    m = (1 - self.p.tail_threshold) / (1 + self.p.tail_power)
    r = (1 - self.p.tail_threshold) / (2 + self.p.tail_power)

    tau_or_0 = -2 * ((1-2*r)/(2-m))**2

    return np.where(
        tau_or_raw >= tau_or_0,
        (tau_or_raw - tau_or_0) / (1.0 - tau_or_0),
        (tau_or_raw - tau_or_0) / (1.0 + tau_or_0)
    )

  def _tau_matrix(self, losses, U):
    losses = np.asarray(losses, dtype=float)
    U = np.asarray(U, dtype=float)

    squeeze = losses.ndim == 2
    if squeeze:
      losses, U = losses[None], U[None]

    B, T_dim, N = losses.shape
    h = self.tail_weight(U)
    n0 = T_dim * (T_dim - 1) / 2.0

    G, Gw_and, Gw_or, zeros, D_and, D_or_u = self._get_tau(losses, h)
    ties = (zeros - T_dim) / 2.0

    n_eff = n0 - ties
    den_g = np.sqrt(n_eff[:, :, None] * n_eff[:, None, :])
    tau_glob = np.divide(G / 2.0, den_g, out=np.zeros((B, N, N)), where=den_g > 0)

    hT = h.swapaxes(1, 2)
    w2 = h ** 2
    w2T = w2.swapaxes(1, 2)
    W_prod = hT @ h
    W2_prod = w2T @ w2
    D_v = D_and.swapaxes(1, 2)
    den_and = np.sqrt(D_and * D_v)

    tau_and = np.divide(
        0.5*Gw_and, den_and, out=np.full((B, N, N), np.nan), where=den_and > 0
    )

    ess_and = np.divide(
        W_prod ** 2,
        W2_prod,
        out=np.zeros((B, N, N)),
        where=W2_prod > 0
    )

    D_or_v = D_or_u.swapaxes(1, 2)

    den_or = np.sqrt(
        np.maximum(D_or_u, 0)
        * np.maximum(D_or_v, 0)
    )

    tau_or_raw = np.divide(
        Gw_or, den_or, out=np.full_like(Gw_or, np.nan), where=den_or > 0
    )

    s1 = h.sum(axis=1)
    s2 = w2.sum(axis=1)
    cross = w2T @ h

    sum_w = s1[:, :, None] + s1[:, None, :] - W_prod
    sum_w2 = (
        s2[:, :, None] + s2[:, None, :]
        + w2T @ w2 + 2*W_prod
        - 2*cross - 2*cross.swapaxes(1, 2)
    )

    ess_or = np.divide(
        sum_w**2,
        sum_w2,
        out=np.zeros_like(sum_w),
        where=sum_w2 > 0
    )
    tau_or = self._adjust_tau_or(tau_or_raw)

    out = (tau_glob, tau_and, tau_or, ess_and, ess_or)
    return tuple(o[0] for o in out) if squeeze else out

  def _get_rel_score(self, ess):
    score = (
      (ess - self.p.min_effective_sample)
      / (ess + self.p.shrink_target_ess - self.p.min_effective_sample)
    )

    score = np.clip(score, 0.0, 1.0)
    return score

  def fit_kendall(self, returns, assets=None, tau_prev=None, return_tau=False):
    if assets is None and hasattr(returns, "columns"):
      assets = list(returns.columns)

    losses = -np.asarray(returns, dtype=float)
    n_steps = losses.shape[-2]

    U = self.empirical_uniforms(losses, T=n_steps)
    tau_glob, tau_and, tau_or, ess_and, ess_or = self._tau_matrix(losses=losses, U=U)

    reliability_score_and = self._get_rel_score(ess_and)
    reliability_score_or = self._get_rel_score(ess_or)

    eff_comp_and = self.p.max_tail_inf * self.p.and_pref * reliability_score_and
    eff_comp_or = self.p.max_tail_inf * (1 - self.p.and_pref) * reliability_score_or
    eff_comp_glob = 1 - eff_comp_and - eff_comp_or

    tau_final = (
        eff_comp_and * tau_and
        + eff_comp_or * tau_or
        + eff_comp_glob * tau_glob
    )

    return tau_final

  def get_tau_dist(self, returns, assets=None, tau_prev=None, return_tau=False):
      return self.fit_kendall(returns=returns, assets=assets,
                              tau_prev=tau_prev, return_tau=return_tau)

## HierarchicalFilterOptimizer



In [ ]:
class HierarchicalFilterOptimizer:
  def __init__(self, params, **kwargs):
    self.fp = params
    super().__init__(params=params, **kwargs)

  def _get_bootstrapped_ari(self, theta_w, distances_boot, distances_true, k):
    theta_w = np.asarray(theta_w, dtype=float)

    comp_boot = np.sum(theta_w.reshape(-1, 1, 1, 1) * distances_boot, axis=0)
    comp_true = np.sum(theta_w.reshape(-1, 1, 1) * distances_true, axis=0)

    ref_labels = cut_tree(
        linkage(squareform(comp_true, checks=False), method="average"), n_clusters=k
    ).ravel()

    aris = []
    for dist_b in comp_boot:
        Z_b = linkage(squareform(dist_b, checks=False), method="average")
        labels_b = cut_tree(Z_b, n_clusters=k).ravel()
        aris.append(adjusted_rand_score(ref_labels, labels_b))

    mean = np.mean(aris)
    std = np.std(aris)
    median = np.median(aris)
    ari_10_pct = np.quantile(aris, 0.1)

    return mean, std, median, ari_10_pct


  def _get_opt_theta(self, distances_boot, distances_true, k, theta0=None):
    theta0 = np.array([0.5, 0.5, 0]) if theta0 is None else np.asarray(theta0, dtype=float)

    def objective(theta_w):
        mean_term, std_term, _, _ = self._get_bootstrapped_ari(
            theta_w=theta_w,
            distances_boot=distances_boot,
            distances_true=distances_true,
            k=k
        )
        reg_term = np.sum((theta_w - theta0) ** 2)

        dist = np.sum(theta_w.reshape(-1, 1, 1, 1) * distances_boot, axis=0)
        Z = linkage(squareform(dist, checks=False), method="average")
        labels = cut_tree(Z, n_clusters=3).ravel()
        score = silhouette_score(dist, labels, metric="precomputed")

        return std

    n = 3

    bounds = Bounds(lb=np.zeros(n), ub=np.ones(n))

    linear_constraint = LinearConstraint(A=np.ones((1, n)), lb=1.0, ub=1.0)
    theta_0 = np.full(n, 1.0 / n)


    result = minimize(
        objective,
        x0=theta0,
        method="SLSQP",
        bounds=bounds,
        constraints=[linear_constraint],
    )

    theta_w = result.x

    mean, std, median, ari_10_pct = self._get_bootstrapped_ari(
        theta_w=theta_w,
        distances_boot=distances_boot,
        distances_true=distances_true,
        k=k
    )

    return theta_w, median, ari_10_pct

## HierarchicalFilterHelpers

In [ ]:
class HierarchicalFilterHelpers(TailWeightedKendall, HierarchicalFilterOptimizer):
    def __init__(self, params, **kwargs):
        self.fp = params
        self.thetas = None
        super().__init__(params=params, **kwargs)

    @staticmethod
    def _validate_distance(dist, name: str = "distance", atol: float = 1e-6) -> None:

        D = np.asarray(dist, dtype=float)

        if D.ndim != 2 or D.shape[0] != D.shape[1]:
            raise ValueError(f"{name} must be a square (N,N) matrix; got shape {D.shape}")

        if not np.isfinite(D).all():
            raise ValueError(f"{name} contains non-finite values")

        if (D < -atol).any():
            raise ValueError(
                f"{name} has negative entries (min {D.min():.3g}); need D_ij >= 0"
            )

        asymmetry = np.abs(D - D.T).max()
        if asymmetry > atol:
            raise ValueError(
                f"{name} is not symmetric (max |D_ij - D_ji| = {asymmetry:.3g}); "
                "need D_ij == D_ji"
            )

        diagonal = np.abs(np.diag(D)).max()
        if diagonal > atol:
            raise ValueError(
                f"{name} has a nonzero diagonal (max |D_ii| = {diagonal:.3g}); "
                "need D_ii == 0"
            )

    def _get_kendall_distance(self, returns, tau_prev=None):
        dist_mtx = self.get_tau_dist(
            returns=returns,
            tau_prev=tau_prev,
            return_tau=False
        )

        return np.asarray(dist_mtx, dtype=float)

    def _get_cosine_similarity(self, betas) -> np.ndarray:
        B = np.asarray(betas, dtype=float)

        if B.ndim not in (2, 3):
            raise ValueError("betas must be (N, F) or (n_boot, N, F)")

        norm = np.linalg.norm(B, axis=-1, keepdims=True)
        B = B / np.where(norm == 0, 1.0, norm)

        return np.clip(B @ np.swapaxes(B, -1, -2), -1.0, 1.0)

    def _get_cvar(self, returns, alpha: float = 0.05) -> np.ndarray:
        if not 0.0 < alpha < 1.0:
            raise ValueError("alpha must be in (0, 1)")

        R = np.asarray(returns, dtype=float)

        if R.ndim not in (2, 3):
            raise ValueError("returns must be (T, N) or (n_boot, T, N)")

        var = np.quantile(R, alpha, axis=-2, keepdims=True)
        mask = R <= var

        count = mask.sum(axis=-2)
        total = np.where(mask, R, 0.0).sum(axis=-2)
        cvar = np.divide(total, count, out=np.full(total.shape, np.nan), where=count > 0)

        if np.isnan(cvar).any():
            raise ValueError("CVaR is undefined (no valid returns) for some assets")

        return cvar

    def _get_co_exceedances(self, returns):
        R = np.asarray(returns, dtype=float)
        cvar = self._get_cvar(returns=R)

        I = (R <= cvar[..., None, :]).astype(float)

        n_ij = np.swapaxes(I, -1, -2) @ I
        n_i = I.sum(axis=-2)

        denominator = np.sqrt(n_i[..., :, None] * n_i[..., None, :])

        lambda_hat = np.divide(n_ij, denominator, out=np.zeros_like(n_ij), where=denominator != 0)

        return 1 - lambda_hat

    def _bootstrap_returns_and_betas(self, returns, betas, bootstrap_params=None, rng=None):
        params = bootstrap_params or BootstrapParams()
        n_assets = returns.shape[1]

        joint = pd.concat([returns, betas], axis=1)
        samples = BlockBootstrap.resample(joint, params, rng=rng)

        n_boot, horizon, _ = samples.shape
        boot_returns = samples[:, :, :n_assets]
        boot_betas = samples[:, :, n_assets:].reshape(n_boot, horizon, n_assets, -1)

        return boot_returns, boot_betas

    def _get_ari_test_params(self, boot_returns, boot_betas, tau_prev):
        boot_returns = np.asarray(boot_returns, dtype=float)
        boot_betas = np.asarray(boot_betas, dtype=float)

        if boot_betas.ndim == 4:
            boot_betas = boot_betas.mean(axis=1)

        n_boot, _, n_assets = boot_returns.shape

        tail = self._get_co_exceedances(boot_returns)
        cos_sim = self._get_cosine_similarity(boot_betas)
        cosine = np.sqrt((1 - cos_sim) / 2)
        kendall = self._get_kendall_distance(returns=boot_returns, tau_prev=tau_prev)

        distances_boot = np.stack([tail, cosine, kendall], axis=0)

        idx = np.arange(n_assets)
        distances_boot[:, :, idx, idx] = 0.0

        names = ("tail", "cosine", "kendall")
        for d, name in enumerate(names):
            for b in range(n_boot):
                self._validate_distance(dist=distances_boot[d, b], name=f"bootstrap {b} {name} distance")

        return distances_boot

    def _get_opt_k(self, distances_true, distances_boot, theta0=None):
        theta0 = np.array([0.5, 0.5, 0]) if theta0 is None else theta0

        n_assets = distances_true.shape[-1]

        k_range = np.arange(1, min(self.fp.max_k, n_assets) + 1)

        silhouette_scores = []
        ari_medians = []
        ari_10_pct_q = []
        thetas = []
        composites = []
        labels_per_k = []

        for k_idx, k in enumerate(k_range):

            theta_w, median, ari_10_pct = self._get_opt_theta(
                distances_boot=distances_boot,
                distances_true=distances_true,
                k=k,
                theta0=theta0
            )

            D = np.sum(theta_w.reshape(-1, 1, 1) * distances_true, axis=0)
            Z = linkage(squareform(D, checks=False), method="average")
            labels = cut_tree(Z, n_clusters=k).ravel()

            silhouette_scores.append(
                silhouette_score(
                    D,
                    labels,
                    metric="precomputed"
                )
            )
            ari_medians.append(median)
            ari_10_pct_q.append(ari_10_pct)
            thetas.append(theta_w)
            composites.append(D)
            labels_per_k.append(labels)

        ari_10_pct_q = np.asarray(ari_10_pct_q, dtype=float)
        ari_medians = np.asarray(ari_medians, dtype=float)
        silhouette_scores = np.asarray(silhouette_scores, dtype=float)

        valid = (
            (ari_10_pct_q > self.fp.min_ari)
            & (silhouette_scores > self.fp.min_silhouette)
        )

        if not valid.any():
            raise ValueError("No valid k found")

        best = np.flatnonzero(valid)[np.argmax(silhouette_scores[valid])]

        optimal_k = int(k_range[best])

        return optimal_k, labels_per_k[best], composites[best], thetas[best]

## HierarchicalFilter

In [ ]:
class HierarchicalFilter(HierarchicalFilterHelpers):
    def get_hierarchical_structure(self, returns, betas, tau_prev=None, bootstrap_params=None):
        n_assets = returns.shape[1]
        betas_mean = betas.to_numpy().reshape(len(betas), n_assets, -1).mean(axis=0)

        cos_sim = self._get_cosine_similarity(betas_mean)
        factor_distance = np.sqrt((1 - cos_sim) / 2)
        self._validate_distance(dist=factor_distance, name="factor distance")

        kendall_distance = self._get_kendall_distance(returns=returns, tau_prev=tau_prev)
        self._validate_distance(dist=kendall_distance, name="Kendall distance")

        tail_dist = self._get_co_exceedances(returns)
        self._validate_distance(dist=tail_dist, name="tail distance")

        distances_true = np.stack([tail_dist, factor_distance, kendall_distance], axis=0)

        boot_returns, boot_betas = self._bootstrap_returns_and_betas(
            returns=returns, betas=betas, bootstrap_params=bootstrap_params
        )
        distances_boot = self._get_ari_test_params(
            boot_returns=boot_returns,
            boot_betas=boot_betas,
            tau_prev=tau_prev
        )

        optimal_k, opt_labels, comp_dist, theta_w = self._get_opt_k(
            distances_true=distances_true,
            distances_boot=distances_boot
        )

        return (optimal_k, opt_labels, comp_dist, theta_w)

# RegimeFilter

In [ ]:
class RegimeDetector:
  pass

In [ ]:
class RegimeFilterHelpers(RegimeDetector):
  pass

In [ ]:
class RegimeFilter(RegimeFilterHelpers):
  pass

# Filter

In [ ]:
class Filter(FactorFilter, HierarchicalFilter, RegimeFilter):
  pass